In [1]:
!pip install -q transformers peft bitsandbytes accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00


In [2]:
import os
import time
import torch
import shutil
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

GPU: Tesla T4
VRAM: 15.64 GB


In [3]:
from google.colab import files
uploaded = files.upload()

Saving tokenizer.json to tokenizer.json
Saving adapter_model.safetensors to adapter_model.safetensors
Saving adapter_config.json to adapter_config.json


In [4]:
os.makedirs("/content/adapters", exist_ok=True)
for f in ["adapter_model.safetensors", "adapter_config.json", "tokenizer.json"]:
    if os.path.exists(f"/content/{f}"):
        shutil.move(f"/content/{f}", f"/content/adapters/{f}")
print(os.listdir("/content/adapters"))

['adapter_model.safetensors', 'tokenizer.json', 'adapter_config.json']


In [5]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer  = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype      = torch.float16,
    device_map = "auto"
)
print("base model loaded")
print("memory:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

base model loaded
memory: 2.2 GB


In [6]:
model = PeftModel.from_pretrained(base_model, "/content/adapters")
model = model.merge_and_unload()
print("merged")
print("memory:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

merged
memory: 2.21 GB


In [7]:
os.makedirs("/content/quantized/model-fp16", exist_ok=True)
model.save_pretrained("/content/quantized/model-fp16")
tokenizer.save_pretrained("/content/quantized/model-fp16")
print("FP16 saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FP16 saved


In [8]:
bnb_int8   = BitsAndBytesConfig(load_in_8bit=True)
model_int8 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config = bnb_int8,
    device_map          = "auto"
)
os.makedirs("/content/quantized/model-int8", exist_ok=True)
model_int8.save_pretrained("/content/quantized/model-int8")
tokenizer.save_pretrained("/content/quantized/model-int8")
print("INT8 saved")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT8 saved


In [9]:
bnb_int4   = BitsAndBytesConfig(
    load_in_4bit           = True,
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_quant_type    = "nf4"
)
model_int4 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config = bnb_int4,
    device_map          = "auto"
)
os.makedirs("/content/quantized/model-int4", exist_ok=True)
model_int4.save_pretrained("/content/quantized/model-int4")
tokenizer.save_pretrained("/content/quantized/model-int4")
print("INT4 saved")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT4 saved


In [10]:
!git clone https://github.com/ggerganov/llama.cpp -q
!pip install -q -r /content/llama.cpp/requirements.txt

from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    filename  = "tokenizer.model",
    local_dir = "/content/quantized/model-fp16"
)

!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/quantized/model-fp16 \
    --outfile /content/quantized/model.gguf \
    --outtype q8_0

print("GGUF size:", round(os.path.getsize("/content/quantized/model.gguf") / 1e9, 2), "GB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 92.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.6/178.6 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 126.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.6 MB/s eta 0:00:00
 

In [11]:
from google.colab import files
files.download("/content/quantized/model.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
def benchmark(model_path, label):
    tok   = AutoTokenizer.from_pretrained(model_path)
    mdl   = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map = "auto",
        dtype      = torch.float16
    )
    prompt = "Write a Python function to check if a number is prime."
    inputs = tok(prompt, return_tensors="pt").to(mdl.device)
    start  = time.time()
    output = mdl.generate(**inputs, max_new_tokens=100)
    end    = time.time()
    tokens = output.shape[1] - inputs["input_ids"].shape[1]
    print(f"{label} → {round(tokens/(end-start), 2)} tokens/sec | VRAM: {round(torch.cuda.memory_allocated()/1e9, 2)} GB")
    del mdl
    torch.cuda.empty_cache()

# copy tokenizer.model to int8 and int4
!cp /content/quantized/model-fp16/tokenizer.model /content/quantized/model-int8/
!cp /content/quantized/model-fp16/tokenizer.model /content/quantized/model-int4/

benchmark("/content/quantized/model-fp16", "FP16")
benchmark("/content/quantized/model-int8", "INT8")
benchmark("/content/quantized/model-int4", "INT4")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FP16 → 21.15 tokens/sec | VRAM: 6.49 GB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INT8 → 8.9 tokens/sec | VRAM: 5.52 GB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INT4 → 23.13 tokens/sec | VRAM: 5.1 GB
